In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
from housepriceprediction.modeling import build_lasso_pipeline
import numpy as np
from housepriceprediction.config import DATA_RAW, RANDOM_STATE
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
import pandas as pd
from sklearn.inspection import permutation_importance
from IPython.display import display

train_df = pd.read_csv(DATA_RAW / "train.csv", keep_default_na=False, na_values=["NA"])
train_df = train_df[train_df["GrLivArea"] <= 4000].reset_index(drop=True)

y = train_df["SalePrice"].apply(np.log1p)
X = train_df.drop(columns=["Id", "SalePrice"])
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
lasso_pipeline = build_lasso_pipeline()
pred_log = cross_val_predict(lasso_pipeline, X, y, cv=cv, n_jobs=-1)
pred_price = np.expm1(pred_log)
residuals = y - pred_log
error_pct = np.expm1(residuals) * 100

errors = pd.DataFrame({
    "Id": train_df["Id"],
    "SalePrice": train_df["SalePrice"],
    "Predicted": np.round(pred_price, 2),
    "Residual": np.round(residuals, 4),
    "ErrorPct": np.round(error_pct, 2),
    "Neighborhood": train_df["Neighborhood"],
    "OverallQual": train_df["OverallQual"],
    "GrLivArea": train_df["GrLivArea"],
    "SaleCondition": train_df["SaleCondition"],
    "YearBuilt": train_df["YearBuilt"],
})

top_10_errors = errors.reindex(errors["Residual"].abs().sort_values(ascending=False).index).head(10)
print("--- Top 10 Houses with the biggest prediction errors (Out-of-Fold) ---")
display(top_10_errors)

--- Top 10 Houses with the biggest prediction errors (Out-of-Fold) ---


,Id,SalePrice,Predicted,Residual,ErrorPct,Neighborhood,OverallQual,GrLivArea,SaleCondition,YearBuilt
631,633,82500,182289.96,-0.7928,-54.74,NWAmes,7,1411,Family,1977
30,31,40000,81747.74,-0.7147,-51.07,IDOTRR,4,1317,Normal,1920
462,463,62383,119763.64,-0.6522,-47.91,Sawyer,5,864,Normal,1965
1320,1325,147000,276301.39,-0.6311,-46.80,Somerst,8,1795,Partial,2006
966,969,37900,69663.43,-0.6087,-45.59,OldTown,3,968,Abnorml,1910
88,89,85000,47992.15,0.5716,77.11,IDOTRR,3,1526,Abnorml,1915
495,496,34900,61565.19,-0.5676,-43.31,IDOTRR,4,720,Abnorml,1920
410,411,60000,100767.03,-0.5185,-40.46,Edwards,5,1276,Abnorml,1958
968,971,135000,80493.51,0.5171,67.71,NAmes,4,1192,Abnorml,1949
587,589,143000,239489.44,-0.5157,-40.29,ClearCr,5,1473,Partial,1968


In [20]:
def error_by_group(errors: pd.DataFrame, column: str, min_count: int = 1, sort_by: str = "Mean_Abs_ErrorPct", ascending: bool = False) -> pd.DataFrame:
    """Calculate error metrics grouped by a specified column.

    Allows sorting either by error metrics or linearly by group index (e.g.,
    price bands).
    """
    df = errors.copy()
    df["AbsErrorPct"] = df["ErrorPct"].abs()
    df["AbsResidual"] = df["Residual"].abs()

    grouped = (
        df.groupby(column, observed=False)
        .agg(
            Count=("Id", "count"),
            Mean_Residual=("Residual", "mean"),
            Mean_Abs_Residual=("AbsResidual", "mean"),
            Mean_Abs_ErrorPct=("AbsErrorPct", "mean"),
        )
        .reset_index()
    )
    grouped = grouped[grouped["Count"] >= min_count]
    grouped = grouped.sort_values(by=sort_by, ascending=ascending).reset_index(drop=True)
    grouped["Mean_Residual"] = grouped["Mean_Residual"].round(4)
    grouped["Mean_Abs_Residual"] = grouped["Mean_Abs_Residual"].round(4)
    grouped["Mean_Abs_ErrorPct"] = grouped["Mean_Abs_ErrorPct"].round(2)
    return grouped


errors["PriceBand"] = pd.qcut(errors["Predicted"], q=5)
errors["ActualPriceBand"] = pd.qcut(errors["SalePrice"], q=5)

predicted_band_errors = error_by_group(errors, column="PriceBand", sort_by="PriceBand", ascending=True)
actual_band_errors = error_by_group(errors, column="ActualPriceBand", sort_by="ActualPriceBand",ascending=True)

print("--- 1. CORRECT: Error Analysis by Predicted Price (PriceBand) ---")
display(predicted_band_errors)

print(
    "\n--- 2. STATISTICAL TRAP: Error Analysis by Actual Sale Price"
    " (ActualPriceBand) ---"
)

display(actual_band_errors)

--- 1. CORRECT: Error Analysis by Predicted Price (PriceBand) ---


,PriceBand,Count,Mean_Residual,Mean_Abs_Residual,Mean_Abs_ErrorPct
0,"(47992.149000000005, 123003.21]",292,-0.0055,0.1100,10.87
1,"(123003.21, 146440.77]",291,0.0100,0.0657,6.61
2,"(146440.77, 181544.2]",291,-0.0051,0.0630,6.25
3,"(181544.2, 226986.99]",291,-0.0113,0.0685,6.75
4,"(226986.99, 601009.67]",291,0.0086,0.0788,7.93



--- 2. STATISTICAL TRAP: Error Analysis by Actual Sale Price (ActualPriceBand) ---


,ActualPriceBand,Count,Mean_Residual,Mean_Abs_Residual,Mean_Abs_ErrorPct
0,"(34899.999, 124000.0]",295,-0.0469,0.1100,10.41
1,"(124000.0, 147000.0]",294,0.0039,0.0698,6.98
2,"(147000.0, 179200.0]",286,0.0080,0.0662,6.70
3,"(179200.0, 230000.0]",294,-0.0034,0.0621,6.21
4,"(230000.0, 625000.0]",287,0.0363,0.0777,8.10


In [21]:
# Error analysis (Lasso, alpha=0.003, out-of-fold predictions)
#
# 1. No systematic bias. Mean residual is close to 0 in every group (SaleCondition,
#    Neighborhood, predicted price band): the model already uses these columns, so it has
#    learned the average price shifts between groups.
#
# 2. What differs is predictability (mean absolute error per house):
#    - SaleCondition: Normal 6.9%, Partial 9.1%, Abnorml / Family / Alloca ~13-14%.
#      Distressed or family sales depend on the seller's situation, not on the house.
#    - Neighborhood: from 4.8% (CollgCr) to 17.3% (IDOTRR). Old, mixed areas (IDOTRR,
#      OldTown, SWISU, Edwards, BrkSide) are hard; newer, homogeneous ones (CollgCr, Gilbert,
#      SawyerW) are easy.
#    - Predicted price: U-shape. Cheapest 20% -> 10.9%, middle -> ~6.5%, most expensive 20% -> 7.9%.
#
# 3. Top 10 errors: 8 of 10 overestimated, mostly Abnorml / Partial / Family sales and old,
#    cheap houses (IDOTRR, OldTown). Id 589 is "Partial" but built in 1968: possibly
#    inconsistent data.
#
# 4. Statistical trap avoided: grouping by the *actual* price shows a fake bias
#    (-4.7% for the cheapest, +3.6% for the most expensive) caused by regression to the mean.
#    Grouping by the *predicted* price (what a user sees) shows no bias.
#
# For the web app: for a normal sale, the typical error is ~7%.

In [22]:
# 1. Fit the Lasso pipeline on the entire dataset (X, y)
lasso_full = build_lasso_pipeline()
lasso_full.fit(X, y)

# 2. Extract feature names from scaler and fitted coefficients from regressor
feature_names = lasso_full.named_steps["scaler"].feature_names_in_
coefficients = lasso_full.named_steps["regressor"].coef_

# 3. Build a DataFrame with feature importances and percentage effects
coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coef": coefficients,
    "EffectPct": (np.expm1(coefficients)) * 100,
})

# 4. Count total features vs. features set to zero by Lasso regularization
total_features = len(coef_df)
zero_features = int((coef_df["Coef"] == 0).astype(int).sum())
active_features = total_features - zero_features

print("--- Lasso Feature Selection Summary ---")
print(f"Total features generated: {total_features}")
print(
    f"Features eliminated (Coef = 0): {zero_features}"
    f" ({zero_features / total_features * 100:.1f}%)"
)
print(f"Active features retained: {active_features}\n")

# 5. Display the top 15 most influential features sorted by absolute coefficient value
top_15_features = (
    coef_df.reindex(coef_df["Coef"].abs().sort_values(ascending=False).index)
    .head(15)
    .reset_index(drop=True)
)

# Format numerical columns for clean display
top_15_features["Coef"] = top_15_features["Coef"].round(4)
top_15_features["EffectPct"] = top_15_features["EffectPct"].round(2)

print("--- Top 15 Most Influential Features (Sorted by Absolute Coef) ---")
display(top_15_features)

--- Lasso Feature Selection Summary ---
Total features generated: 214
Features eliminated (Coef = 0): 132 (61.7%)
Active features retained: 82

--- Top 15 Most Influential Features (Sorted by Absolute Coef) ---


,Feature,Coef,EffectPct
0,GrLivArea,0.0745,7.74
1,TotalSF,0.0740,7.68
2,OverallQual,0.0708,7.33
3,YearBuilt,0.0408,4.17
4,OverallCond,0.0356,3.62
5,MSZoning_C (all),-0.0287,-2.83
6,BsmtFinSF1,0.0277,2.80
7,GarageCars,0.0197,1.99
8,Neighborhood_Crawfor,0.0197,1.99
9,Functional,0.0197,1.98


In [23]:
# 1. Extract feature scales (standard deviations) from the fitted StandardScaler
scales = lasso_full.named_steps["scaler"].scale_

# 2. Add Scale and CoefPerUnit columns to our existing coef_df DataFrame
coef_df["Scale"] = scales
coef_df["CoefPerUnit"] = coef_df["Coef"] / coef_df["Scale"]

# 3. Define human-interpretable features and their practical domain units
unit_definitions = [
    ("OverallQual", 1, "+1 quality point (scale 1-10)"),
    ("OverallCond", 1, "+1 condition point (scale 1-10)"),
    ("GrLivArea", 100, "+100 sq ft above grade living area"),
    ("TotalSF", 100, "+100 sq ft total area (living + basement)"),
    ("YearBuilt", 10, "+10 years newer construction"),
    ("GarageCars", 1, "+1 garage car capacity"),
]

# 4. Compute practical percentage effects for selected continuous features
practical_effects = []
for feature_name, unit_size, unit_description in unit_definitions:
    # Locate the feature in coef_df
    feature_row = coef_df[coef_df["Feature"] == feature_name]

    if not feature_row.empty:
        coef_per_unit = feature_row["CoefPerUnit"].values[0]
        total_unit_coef = coef_per_unit * unit_size
        pct_effect = (np.expm1(total_unit_coef)) * 100

        practical_effects.append({
            "Feature": feature_name,
            "Unit": unit_description,
            "EffectPct (%)": round(float(pct_effect), 2),
        })

# 5. Display the final practical feature effects table
practical_df = pd.DataFrame(practical_effects)

print("--- Practical Feature Effects in Real Domain Units ---")
display(practical_df)

--- Practical Feature Effects in Real Domain Units ---


,Feature,Unit,EffectPct (%)
0,OverallQual,+1 quality point (scale 1-10),5.31
1,OverallCond,+1 condition point (scale 1-10),3.25
2,GrLivArea,+100 sq ft above grade living area,1.51
3,TotalSF,+100 sq ft total area (living + basement),0.98
4,YearBuilt,+10 years newer construction,1.36
5,GarageCars,+1 garage car capacity,2.67


In [24]:
# 1. Split data into train and test sets (evaluating importance on unseen test data)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

# 2. Fit the Lasso pipeline on the training split
lasso_split = build_lasso_pipeline()
lasso_split.fit(X_train, y_train)

# 3. Compute permutation importance on the test set
# Using negative RMSE so that higher importance values represent a larger drop in performance
perm_result = permutation_importance(
    lasso_split,
    X_test,
    y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

# 4. Build a summary DataFrame for original features
perm_importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance_Mean": perm_result.importances_mean,
    "Importance_Std": perm_result.importances_std,
})

# 5. Sort by highest importance first and format for display
top_15_perm = (
    perm_importance_df.sort_values(by="Importance_Mean", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

top_15_perm["Importance_Mean"] = top_15_perm["Importance_Mean"].round(5)
top_15_perm["Importance_Std"] = top_15_perm["Importance_Std"].round(5)

print("--- Top 15 Original Features by Permutation Importance (Test Set) ---")
display(top_15_perm)

--- Top 15 Original Features by Permutation Importance (Test Set) ---


,Feature,Importance_Mean,Importance_Std
0,GrLivArea,0.04976,0.00296
1,OverallQual,0.04462,0.00341
2,YearBuilt,0.01964,0.00174
3,MSZoning,0.01588,0.00098
4,1stFlrSF,0.01551,0.00215
5,TotalBsmtSF,0.01498,0.00078
6,OverallCond,0.01209,0.00148
7,2ndFlrSF,0.00952,0.00139
8,BsmtFinSF1,0.00848,0.00104
9,Neighborhood,0.00714,0.00130


In [ ]:
# What drives the price (Lasso, alpha=0.003)
#
# Lasso kept 82 of 214 columns (132 coefficients exactly 0).
#
# Practical effects (one feature changed, the others fixed; effects multiply):
#   +1 OverallQual point -> +5.3%   |  +1 OverallCond point -> +3.3%
#   +1 garage car space  -> +2.7%   |  +10 years newer       -> +1.4%
#   +100 sq ft living area -> ~+2.5% in total (GrLivArea +1.5% and TotalSF +1.0% move together)
# Negative: commercial zoning (MSZoning C) -2.8%, medium-density zoning (RM) -1.7%,
#           abnormal sale -1.6%.
#
# Permutation importance (held-out 20%, original columns, increase in RMSE on log):
#   - Size dominates: GrLivArea, 1stFlrSF, TotalBsmtSF, 2ndFlrSF, BsmtFinSF1 together ~0.10.
#   - OverallQual alone 0.045, then YearBuilt, MSZoning, OverallCond.
#   - Neighborhood only 0.007: location matters, but most of it is already carried by
#     size, quality, age and zoning (good neighborhoods have bigger, newer, better houses).
# SHAP skipped: for a linear model it equals coef x (value - mean), i.e. the coefficients above.